# Topic-Level Difficulty Analysis

This notebook analyzes the actual topic-level records in the raw dataset and inspects the generated student-topic analysis dataset. Early features use weeks 1-6 only; future weeks 7-10 are used only for descriptive future performance and target construction.

The student-level Random Forest predicts overall future academic difficulty. This topic-level analysis identifies specific topics needing attention; it is not a topic-level ML model. Low observed performance is reported as an observed performance pattern, not proof that a topic caused difficulty.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
from topic_analysis import EARLY_WEEKS, FUTURE_WEEKS, TARGET_LABELS

RAW_PATH = PROJECT_ROOT / 'data' / 'raw' / 'student_learning_data.csv'
TOPIC_DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'student_topic_early_prediction.csv'
SUMMARY_PATH = PROJECT_ROOT / 'data' / 'processed' / 'topic_summary.csv'
raw = pd.read_csv(RAW_PATH)
topic_data = pd.read_csv(TOPIC_DATA_PATH)
topic_summary = pd.read_csv(SUMMARY_PATH)
sns.set_theme(style='whitegrid', context='notebook')
print('Raw shape:', raw.shape)
print('Topic-level shape:', topic_data.shape)
print('Subjects:', raw['subject'].nunique())
print('Topics:', raw['topic'].nunique())

## 1. Future Distribution and Target Rationale

Before constructing the topic target, the future `topic_score` distribution was inspected. The observed student-topic future mean score percentiles define the target thresholds rather than copying the student-level target.

In [ ]:
future = raw[raw['week'].between(7, 10)].copy()
future_topic = future.groupby(['student_id', 'subject', 'topic']).agg(
    future_mean_topic_score=('topic_score', 'mean'),
).reset_index()
future_weekly = future.groupby(['student_id', 'subject', 'topic', 'week'])['topic_score'].mean().unstack()
future_topic['future_topic_score_change'] = (future_weekly[10] - future_weekly[7]).to_numpy()
print('Raw future topic-score distribution:')
display(future['topic_score'].describe(percentiles=[.10, .25, .50, .75, .90]).to_frame())
print('Future student-topic mean distribution:')
display(future_topic[['future_mean_topic_score', 'future_topic_score_change']].describe(percentiles=[.10, .25, .50, .75, .90]))

plt.figure(figsize=(9, 5))
sns.histplot(future_topic['future_mean_topic_score'], kde=True, color='#4472C4')
plt.title('Future student-topic mean score distribution')
plt.xlabel('Mean topic score, weeks 7-10')
plt.show()

### Empirical topic status rule

- **High attention (2):** future mean score at or below the observed 10th percentile, or future mean at or below the observed 25th percentile with a week-7-to-week-10 decline at or below the observed 25th percentile of change.
- **Moderate attention (1):** remaining future mean scores at or below the observed median, or any score with a decline at or below the observed 25th percentile of change.
- **Low attention (0):** all other observed future outcomes.

This is a descriptive prototype status based on the available `topic_score`; it is not a causal difficulty label.

## 2. Student-Topic Early Features and Leakage Checks

In [ ]:
key_columns = ['student_id', 'subject', 'topic', 'early_weeks_observed', 'average_early_topic_score', 'early_topic_score_trend', 'early_topic_score_standard_deviation', 'average_early_quiz_score', 'average_early_practice_hours', 'total_early_questions_attempted', 'total_early_questions_correct', 'early_question_accuracy', 'future_mean_topic_score', 'future_topic_score_change', 'topic_difficulty']
display(topic_data[key_columns].head(10))
feature_columns = [column for column in topic_data.columns if column not in {'student_id', 'subject', 'topic', 'future_mean_topic_score', 'future_topic_score_change', 'topic_difficulty'}]
print('Early feature count:', len(feature_columns))
print('No target/future fields in features:', not set(['future_mean_topic_score', 'future_topic_score_change', 'topic_difficulty']).intersection(feature_columns))
print('student_id excluded from features:', 'student_id' not in feature_columns)
print('Duplicate student-topic combinations:', topic_data[['student_id', 'subject', 'topic']].duplicated().sum())
print('Every student-topic has six early weeks:', topic_data['early_weeks_observed'].eq(6).all())
print('Early weeks used:', '1-6')
print('Future weeks used for target:', '7-10')

## 3. Topic and Subject Performance

In [ ]:
print('Topics needing attention, ordered by percentage:')
display(topic_summary.sort_values('percentage_needing_attention', ascending=False))

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
ordered = topic_summary.sort_values('average_future_score')
sns.barplot(data=ordered, y='topic', x='average_future_score', hue='subject', dodge=False, errorbar=None, ax=axes[0])
axes[0].set_title('Average future score by topic')
subject_summary = topic_summary.groupby('subject', as_index=False)['average_future_score'].mean().sort_values('average_future_score')
sns.barplot(data=subject_summary, y='subject', x='average_future_score', color='#70AD47', ax=axes[1])
axes[1].set_title('Average future score by subject')
plt.tight_layout()
plt.show()

heatmap_data = topic_summary.pivot(index='subject', columns='topic', values='average_future_score')
plt.figure(figsize=(16, 5))
sns.heatmap(heatmap_data, annot=True, fmt='.1f', cmap='YlGnBu')
plt.title('Future topic performance heatmap')
plt.show()

## 4. Observed Topic Patterns

In [ ]:
topic_patterns = topic_summary.assign(
    performance_rank=topic_summary['average_future_score'].rank(method='min'),
).sort_values('average_future_score')
print('Lower observed future performance:')
display(topic_patterns.head(5)[['subject', 'topic', 'average_future_score', 'percentage_needing_attention']])
print('Observed declining mean trajectories:')
display(topic_summary.sort_values('average_future_score_change').head(5)[['subject', 'topic', 'average_future_score_change']])
print('Observed improvement in mean trajectories:')
display(topic_summary.sort_values('average_future_score_change', ascending=False).head(5)[['subject', 'topic', 'average_future_score_change']])
print('Highest future-score variability:')
display(topic_summary.sort_values('future_score_standard_deviation', ascending=False).head(5)[['subject', 'topic', 'future_score_standard_deviation']])
print('These are observed performance patterns; a low mean does not by itself prove a topic is intrinsically difficult.')

## 5. Student-Topic Example Report

In [ ]:
example = topic_data.sort_values(['topic_difficulty', 'future_mean_topic_score'], ascending=[False, True]).head(1)
example_report = example.rename(columns={
    'student_id': 'Student', 'subject': 'Subject', 'topic': 'Topic',
    'average_early_topic_score': 'Early performance',
    'future_mean_topic_score': 'Future performance',
    'early_topic_score_trend': 'Trend',
    'topic_difficulty': 'Topic status',
})[['Student', 'Subject', 'Topic', 'Early performance', 'Future performance', 'Trend', 'Topic status']]
example_report['Topic status'] = example_report['Topic status'].map({0: 'Low attention', 1: 'Moderate attention', 2: 'High attention'})
display(example_report)

## Conclusion

Topic-level data available: **YES**. The raw data contains 500 students, 5 subjects, 25 actual topics, and complete student-topic observations across weeks 1-10.

A topic-level target is justified for this prototype because `topic_score` is directly observed for each student-topic-week and the weeks 7-10 distribution supports transparent empirical thresholds. The target is a future performance/status category, not a claim of intrinsic topic difficulty and not a copy of the student-level `academic_difficulty` label.

The student-level model remains an overall academic-difficulty predictor. This dataset currently supports topic-level analysis and support prioritization; no separate topic-level ML model is trained here.